# end-grad-default-ones-like composite — cx25: seed reverse pass with ones_like when end_grad is None

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `end-grad-default-ones-like`, `backprop-pop-outgrad-loop`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "end-grad-default-ones-like"
DD_ATOM_IDS = ["end-grad-default-ones-like", "backprop-pop-outgrad-loop"]
DD_SUBTOPICS = ["Backprop: end-grad ones_like default", "Backprop: backprop pop-outgrad loop"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

ARENA's `backprop(end_node, end_grad, sorted_graph, back_funcs)` is the reverse-pass driver. Before the loop can start, you need a seed gradient for `end_node`. If the caller passed `end_grad=None`, the convention is `torch.ones_like(end_node.array)` — interpreted as `d(end_node.sum())/d(end_node)`. Once seeded, the loop pops each node's accumulated grad, dispatches the per-arg back_fn, and accumulates into each parent's slot.

Composing them in one function exercises BOTH atoms: you must default the seed AND drive the reverse loop, with leaves landing in `.grad` and intermediates living in a scratch dict.

### Composite Exercise — seed reverse pass with ones_like when end_grad is None

**Atoms exercised together**: `end-grad-default-ones-like`, `backprop-pop-outgrad-loop`

Implement `cx25_backprop(end_node, end_grad, sorted_graph, back_funcs)` that:

1. **Seeds** the reverse pass — if `end_grad is None`, default to `t.ones_like(end_node.array)`. Otherwise unbox `end_grad.array` (asserting shape matches `end_node.array`).
2. **Drives** the reverse loop — for each node in `sorted_graph`, pop the accumulated grad, dispatch `back_funcs[(recipe.func, argnum)]` for each parent in `recipe.parents`, and accumulate into the parent's slot. Leaves (no recipe) get `.grad` populated/accumulated.

In [ ]:
def cx25_backprop(end_node, end_grad, sorted_graph, back_funcs):
    # Atom A: seed default ← ones_like(end_node.array) when end_grad is None.
    if end_grad is None:
        seed = t.ones_like(end_node.array)
    else:
        assert end_grad.array.shape == end_node.array.shape, (
            f'end_grad shape {tuple(end_grad.array.shape)} mismatches '
            f'end_node shape {tuple(end_node.array.shape)}'
        )
        seed = end_grad.array
    # Atom B: reverse-pass loop — pop accumulated grad, dispatch per parent.
    grads = {id(end_node): seed}
    for node in sorted_graph:
        nid = id(node)
        if nid not in grads:
            continue
        grad_out = grads.pop(nid)
        if node.recipe is None:
            if node.grad is None:
                node.grad = grad_out
            else:
                node.grad = node.grad + grad_out
            continue
        for argnum, parent in node.recipe.parents.items():
            back_fn = back_funcs[(node.recipe.func, argnum)]
            gp = back_fn(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
            pid = id(parent)
            grads[pid] = grads.get(pid, 0) + gp


<details><summary>Show solution — cx25</summary>

```python
def cx25_backprop(end_node, end_grad, sorted_graph, back_funcs):
    # Atom A: seed default ← ones_like(end_node.array) when end_grad is None.
    if end_grad is None:
        seed = t.ones_like(end_node.array)
    else:
        assert end_grad.array.shape == end_node.array.shape, (
            f'end_grad shape {tuple(end_grad.array.shape)} mismatches '
            f'end_node shape {tuple(end_node.array.shape)}'
        )
        seed = end_grad.array
    # Atom B: reverse-pass loop — pop accumulated grad, dispatch per parent.
    grads = {id(end_node): seed}
    for node in sorted_graph:
        nid = id(node)
        if nid not in grads:
            continue
        grad_out = grads.pop(nid)
        if node.recipe is None:
            if node.grad is None:
                node.grad = grad_out
            else:
                node.grad = node.grad + grad_out
            continue
        for argnum, parent in node.recipe.parents.items():
            back_fn = back_funcs[(node.recipe.func, argnum)]
            gp = back_fn(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
            pid = id(parent)
            grads[pid] = grads.get(pid, 0) + gp
```

The seed step and the loop step are inseparable in real ARENA code — the loop can't start without a seed in `grads`, and the seed has no purpose without the loop. `ones_like` is the correct default because the implicit reduction for a non-scalar end_node is `.sum()`, whose Jacobian is exactly ones.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx25'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx25',
        'subtopics': ["Backprop: end-grad ones_like default", "Backprop: backprop pop-outgrad loop"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()